# **Creating partitions from MNIST dataset**

This notebook creates 10 IID part from the MNIST dataset for simulating and testing federated learning algorithms.

In [1]:
import numpy as np
import struct
from array import array

MNIST_PATH = "../../02_data/mnist"
NUM_CLIENTS = 10

## Read data

In [2]:
def read_images_labels(images_filepath, labels_filepath):        
    labels = []
    with open(labels_filepath, 'rb') as file:
        magic, size = struct.unpack(">II", file.read(8))
        if magic != 2049:
            raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
        labels = array("B", file.read())        
    
    with open(images_filepath, 'rb') as file:
        magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
        if magic != 2051:
            raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
        image_data = array("B", file.read())        
    images = []
    for i in range(size):
        images.append([0] * rows * cols)
    for i in range(size):
        img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
        img = img.reshape(28, 28)
        images[i] = img            
    
    return images, labels

train_images, train_labels = read_images_labels(f"{MNIST_PATH}/train-images.idx3-ubyte", f"{MNIST_PATH}/train-labels.idx1-ubyte")
test_images, test_labels = read_images_labels(f"{MNIST_PATH}/t10k-images.idx3-ubyte", f"{MNIST_PATH}/t10k-labels.idx1-ubyte")

## Partitioning data

In [3]:
train_data_to_client_assignment = np.random.uniform(0, NUM_CLIENTS, size=len(train_images)).astype(int)

client_train_images = [[] for _ in range(NUM_CLIENTS)]
client_train_labels = [[] for _ in range(NUM_CLIENTS)]
for i in range(len(train_images)):
    client_id = train_data_to_client_assignment[i]
    client_train_images[client_id].append(train_images[i])
    client_train_labels[client_id].append(train_labels[i])

## Save client data

In [4]:
for client_id in range(NUM_CLIENTS):
    np.save(f"{MNIST_PATH}/client_data/client_{client_id}_train_images.npy", np.array(client_train_images[client_id]))
    np.save(f"{MNIST_PATH}/client_data/client_{client_id}_train_labels.npy", np.array(client_train_labels[client_id]))

## Save test data

In [5]:
np.save(f"{MNIST_PATH}/test_images.npy", np.array(test_images))
np.save(f"{MNIST_PATH}/test_labels.npy", np.array(test_labels))